# SECTION 1: SETUP AND INITIAL DATA EXPLORATION

In [ ]:
# =============================================================================
# SECTION 1: SETUP AND INITIAL DATA EXPLORATION
# =============================================================================

# --- 1.1 Import Core Libraries ---
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

# Import machine learning libraries
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler 

# --- 1.2 Load Datasets ---
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'notebooks' and (NOTEBOOK_DIR / 'notebooks').exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'notebooks'
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = (NOTEBOOK_DIR / '..' / 'data' / 'raw').resolve()
FIGURES_DIR = (NOTEBOOK_DIR / '..' / 'outputs' / 'figures').resolve()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

floor_files = {
    f'Floor{i}': DATA_DIR / f'Floor{i}.csv' for i in range(1, 8)
}

all_floors = {}
for floor_name, file_name in floor_files.items():
    try:
        all_floors[floor_name] = pd.read_csv(file_name)
        print(f"Successfully loaded {file_name}")
    except FileNotFoundError:
        print(f"Error: {file_name} not found.")

# --- 1.3 Initial Data Inspection and Missing Data Visualization for All Floors ---
for floor_name, df in all_floors.items():
    print(f"\n--- Analyzing {floor_name} ---")
    df.info()

    plt.figure(figsize=(15, 6))
    sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
    plt.title(f'Missing Data Visualization for {floor_name}')
    
    # Save generated figures outside the raw data directory.
    save_path = FIGURES_DIR / f'{floor_name}_missing_data_visualization.png'
    plt.savefig(save_path, dpi=600, bbox_inches='tight')
    print(f"Saved figure to: {save_path}")
    
    plt.show()
    plt.close()


In [ ]:
from src.utils import enable_mps_acceleration
enable_mps_acceleration()


In [ ]:
# =============================================================================
# SECTION 1: SETUP AND INITIAL DATA EXPLORATION
# =============================================================================

# --- 1.1 Import Core Libraries ---
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

# Import machine learning libraries
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler 

# --- 1.2 Load Datasets ---
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'notebooks' and (NOTEBOOK_DIR / 'notebooks').exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'notebooks'
DATA_DIR = (NOTEBOOK_DIR / '..' / 'data' / 'raw').resolve()
floor_files = {f'Floor{i}': DATA_DIR / f'Floor{i}.csv' for i in range(1, 8)}
all_floors = {}
for floor_name, file_name in floor_files.items():
    try:
        all_floors[floor_name] = pd.read_csv(file_name)
        print(f"Successfully loaded {file_name}")
    except FileNotFoundError:
        print(f"Error: {file_name} not found.")

# --- 1.3 Initial Data Inspection ---
TARGET_FLOOR = 'Floor6'
df = all_floors[TARGET_FLOOR].copy()
print(f"\n--- Analyzing {TARGET_FLOOR} ---")
df.info()

# --- 1.4 Visualize Missing Data ---
plt.figure(figsize=(15, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title(f'Missing Data Visualization for {TARGET_FLOOR}')
plt.show()

# PROJECT UTILITIES AND CUSTOM FUNCTIONS

In [ ]:
# =============================================================================
# PROJECT UTILITIES AND CUSTOM FUNCTIONS
# =============================================================================

# --- Import Utility and Modeling Libraries ---
import os
import ssl
import sys
import time
import warnings
from pathlib import Path

import certifi
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import setuptools
import tensorflow as tf
from meteostat import Base, Hourly, Point

# --- Publication Figure Utilities and Style Configuration ---
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'notebooks' and (NOTEBOOK_DIR / 'notebooks').exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'notebooks'
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
HYPERPARAMETER_TUNING_DIR = (NOTEBOOK_DIR / 'hyperparameter_tuning').resolve()
HYPERPARAMETER_TUNING_DIR.mkdir(parents=True, exist_ok=True)

from src.utils import (
    evaluate_model,
    quantile_loss,
    reset_figure_manifest,
    save_metrics,
    save_publication_fig,
)

manifest_path = reset_figure_manifest()
print(f'Reset figure manifest at: {manifest_path}')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 11,
    'figure.titlesize': 14,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.frameon': False,
    'savefig.facecolor': 'white',
    'axes.prop_cycle': plt.cycler(color=['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9', '#000000'])
})

# --- Meteostat and SSL Configuration ---
os.environ['SSL_CERT_FILE'] = certifi.where()
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())
Base.cache_dir = str((PROJECT_ROOT / '.meteostat').resolve())
Path(Base.cache_dir).mkdir(parents=True, exist_ok=True)
WEATHER_LOCATION = Point(13.7563, 100.5018)

# --- Global Seeds for Reproducibility ---
tf.random.set_seed(42)
np.random.seed(42)

# --- Sequence Creation Function ---
def create_sequences(features, target, time_steps):
    """Converts arrays of features and a target into sequences for time-series models."""
    X, y = [], []
    for i in range(len(features) - time_steps):
        X.append(features[i:(i + time_steps)])
        y.append(target[i + time_steps])
    return np.array(X), np.array(y)


def aggregate_hourly_sensor_mean(df_floor, sensor_columns, feature_name):
    """Aggregate zone-level hourly indoor sensors to a floor-level average using np.nanmean."""
    if not sensor_columns:
        return pd.Series(np.nan, index=df_floor.resample('h').mean().index, name=feature_name)

    hourly_sensor_frame = df_floor[sensor_columns].resample('h').mean()
    sensor_values = hourly_sensor_frame.to_numpy(dtype=float)

    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        aggregated = np.nanmean(sensor_values, axis=1)

    aggregated[np.all(np.isnan(sensor_values), axis=1)] = np.nan
    return pd.Series(aggregated, index=hourly_sensor_frame.index, name=feature_name)


def fetch_hourly_weather(hourly_index):
    """Fetch Meteostat weather data and align it to the requested hourly index."""
    weather_df = Hourly(
        WEATHER_LOCATION,
        hourly_index.min().to_pydatetime(),
        hourly_index.max().to_pydatetime(),
    ).fetch()

    weather_columns = ['temp', 'rhum', 'wspd']
    missing_weather_columns = [col for col in weather_columns if col not in weather_df.columns]
    if missing_weather_columns:
        raise ValueError(f'Weather data is missing expected columns: {missing_weather_columns}')

    weather_df = weather_df[weather_columns].copy()
    if getattr(weather_df.index, 'tz', None) is not None:
        weather_df.index = weather_df.index.tz_convert('Asia/Bangkok').tz_localize(None)
    weather_df = weather_df.reindex(hourly_index)
    weather_df = weather_df.interpolate(method='time').ffill().bfill()
    weather_df.rename(
        columns={
            'temp': 'Outdoor_Temp',
            'rhum': 'Outdoor_RH',
            'wspd': 'Outdoor_WindSpeed',
        },
        inplace=True,
    )
    return weather_df


def build_hourly_floor_dataframe(raw_floor_df):
    """Apply the exact multivariate preprocessing recipe to a floor dataframe."""
    df_floor = raw_floor_df.copy()
    df_floor['Date'] = pd.to_datetime(df_floor['Date'])
    df_floor.set_index('Date', inplace=True)

    energy_columns = [col for col in df_floor.columns if col.endswith('(kW)')]
    df_energy = df_floor[energy_columns].copy()
    print(f'Missing values before imputation: {df_energy.isna().sum().sum()}')
    df_energy.ffill(inplace=True)
    df_energy.bfill(inplace=True)
    print(f'Missing values after imputation: {df_energy.isna().sum().sum()}')

    df_energy['Total_kW'] = df_energy.sum(axis=1)
    df_hourly = df_energy[['Total_kW']].resample('h').mean()
    df_hourly.rename(columns={'Total_kW': 'Total_kWh'}, inplace=True)

    weather_df = fetch_hourly_weather(df_hourly.index)

    indoor_temp_columns = [col for col in df_floor.columns if col.endswith('(degC)')]
    indoor_rh_columns = [col for col in df_floor.columns if col.endswith('(RH%)')]
    indoor_lux_columns = [col for col in df_floor.columns if col.endswith('(lux)')]

    indoor_features_df = pd.DataFrame(index=df_hourly.index)
    indoor_features_df['Indoor_Temp_Avg'] = aggregate_hourly_sensor_mean(df_floor, indoor_temp_columns, 'Indoor_Temp_Avg')
    indoor_features_df['Indoor_RH_Avg'] = aggregate_hourly_sensor_mean(df_floor, indoor_rh_columns, 'Indoor_RH_Avg')
    indoor_features_df['Indoor_Lux_Avg'] = aggregate_hourly_sensor_mean(df_floor, indoor_lux_columns, 'Indoor_Lux_Avg')

    indoor_features_df['Indoor_Temp_mask'] = indoor_features_df['Indoor_Temp_Avg'].notna().astype(int)
    indoor_features_df['Indoor_RH_mask'] = indoor_features_df['Indoor_RH_Avg'].notna().astype(int)
    indoor_features_df['Indoor_Lux_mask'] = indoor_features_df['Indoor_Lux_Avg'].notna().astype(int)

    indoor_features_df['Indoor_Temp_Avg'] = indoor_features_df['Indoor_Temp_Avg'].fillna(0.0)
    indoor_features_df['Indoor_RH_Avg'] = indoor_features_df['Indoor_RH_Avg'].fillna(0.0)
    indoor_features_df['Indoor_Lux_Avg'] = indoor_features_df['Indoor_Lux_Avg'].fillna(0.0)

    df_hourly = df_hourly.join(weather_df, how='left')
    df_hourly = df_hourly.join(indoor_features_df, how='left')

    return df_hourly, df_floor


# --- Custom Keras Layer: Positional Encoding for Transformers ---
class PositionalEncoding(tf.keras.layers.Layer):
    """
    Custom Keras layer to add positional information to the input embeddings.
    This is essential for the Transformer model to understand sequence order.
    """
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.pos_encoding = self.create_positional_encoding(max_len, d_model)

    def get_angles(self, pos, i, d_model):
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
        return pos * angle_rates

    def create_positional_encoding(self, position, d_model):
        angle_rads = self.get_angles(np.arange(position)[:, np.newaxis],
                                     np.arange(d_model)[np.newaxis, :],
                                     d_model)
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
        pos_encoding = angle_rads[np.newaxis, ...]
        return tf.cast(pos_encoding, dtype=tf.float32)

    def call(self, inputs):
        return inputs + self.pos_encoding[:, :tf.shape(inputs)[1], :]


# SECTION 2: DATA CLEANING AND PREPROCESSING

In [ ]:
# =============================================================================
# SECTION 2: DATA CLEANING AND PREPROCESSING
# =============================================================================

# --- 2.1 Build the Floor 6 Hourly Forecasting Table ---
TARGET_FLOOR = 'Floor6'
df = all_floors[TARGET_FLOOR].copy()
df_hourly, df_floor_indexed = build_hourly_floor_dataframe(df)

print('Merged outdoor weather features and indoor sensing masks into df_hourly.')
print(df_hourly[[
    'Outdoor_Temp', 'Outdoor_RH', 'Outdoor_WindSpeed',
    'Indoor_Temp_Avg', 'Indoor_RH_Avg', 'Indoor_Lux_Avg',
    'Indoor_Temp_mask', 'Indoor_RH_mask', 'Indoor_Lux_mask'
]].head())

# --- 2.2 Split Data into Training, Validation, and Testing Sets ---
n_total = len(df_hourly)
train_end_idx = int(n_total * 0.70)
val_end_idx = int(n_total * 0.85)

train_df = df_hourly.iloc[:train_end_idx].copy()
val_df = df_hourly.iloc[train_end_idx:val_end_idx].copy()
test_df = df_hourly.iloc[val_end_idx:].copy()

split_summary = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Start': [train_df.index.min(), val_df.index.min(), test_df.index.min()],
    'End': [train_df.index.max(), val_df.index.max(), test_df.index.max()],
    'Rows': [len(train_df), len(val_df), len(test_df)]
})
print("\nChronological split summary:")
print(split_summary)
print(f"\nTraining set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Testing set shape: {test_df.shape}")

# --- 2.3 Advanced Outlier Detection with Isolation Forest ---
iso_forest = IsolationForest(contamination=0.01, random_state=42)
train_df['outlier'] = iso_forest.fit_predict(train_df[['Total_kWh']])
val_df['outlier'] = iso_forest.predict(val_df[['Total_kWh']])
test_df['outlier'] = iso_forest.predict(test_df[['Total_kWh']])

capping_value = train_df['Total_kWh'].quantile(0.995)
print(f"\nTraining outliers will be capped at: {capping_value:.2f} kWh")
print('Validation and test targets remain fully uncapped for rigorous evaluation.')

train_df['Total_kWh'] = np.where(train_df['outlier'] == -1, capping_value, train_df['Total_kWh'])

train_df.drop(columns=['outlier'], inplace=True)
val_df.drop(columns=['outlier'], inplace=True)
test_df.drop(columns=['outlier'], inplace=True)

print('Training-only outlier handling complete.')

plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
sns.boxplot(y=df_hourly['Total_kWh'])
plt.title('Hourly Total kWh (Before Outlier Handling)')

plt.subplot(1, 2, 2)
sns.boxplot(y=pd.concat([train_df, val_df, test_df])['Total_kWh'])
plt.title('Hourly Total kWh (After Train-Only Outlier Handling)')
plt.tight_layout()
save_publication_fig('Fig01', 'outlier_handling_comparison')
plt.show()

# SECTION 3: FEATURE ENGINEERING AND EXPLORATORY DATA ANALYSIS (EDA)

In [ ]:
# =============================================================================
# SECTION 3: FEATURE ENGINEERING AND EXPLORATORY DATA ANALYSIS (EDA)
# =============================================================================

# --- 3.1 Create Time-Based Features ---
# We create features from the datetime index to capture temporal patterns.

def create_features(df):
    """Creates time series features from a datetime index."""
    df = df.copy()
    df['Hour'] = df.index.hour
    df['DayOfWeek'] = df.index.dayofweek  # Monday=0, Sunday=6
    df['Month'] = df.index.month
    df['WeekOfYear'] = df.index.isocalendar().week.astype(int) # Ensure integer type
    df['IsWeekend'] = (df.index.dayofweek >= 5).astype(int)
    return df

train_df = create_features(train_df)
val_df = create_features(val_df)
test_df = create_features(test_df)

print("Added temporal features (Hour, DayOfWeek, etc.).")

# --- 3.2 Create Holiday Feature ---
# Incorporating known holidays can significantly improve model accuracy.
holidays = pd.to_datetime([
    '2018-07-27', '2018-07-30', '2018-08-13', '2018-10-23', '2018-12-05',
    '2018-12-10', '2018-12-31', '2019-01-01', '2019-02-19', '2019-04-08',
    '2019-04-13', '2019-04-14', '2019-04-15', '2019-05-06', '2019-05-18',
    '2019-07-16', '2019-07-28', '2019-08-12', '2019-10-14', '2019-10-23',
    '2019-12-05', '2019-12-10', '2019-12-31'
])

# We use .normalize() to get the date at midnight and then use .isin()
# This keeps the index as a DatetimeIndex, which has the .isin() method.
train_df['IsHoliday'] = train_df.index.normalize().isin(holidays).astype(int)
val_df['IsHoliday'] = val_df.index.normalize().isin(holidays).astype(int)
test_df['IsHoliday'] = test_df.index.normalize().isin(holidays).astype(int)

print("Added 'IsHoliday' feature.")
print(f"Found {train_df['IsHoliday'].sum()} holiday hours in the training set.")
print(f"Found {val_df['IsHoliday'].sum()} holiday hours in the validation set.")
print(f"Found {test_df['IsHoliday'].sum()} holiday hours in the testing set.")


# --- 3.3 Cyclical Feature Encoding ---
# This helps models understand the cyclical nature of time (e.g., December is close to January).
def encode_cyclical(df, col, max_val):
    df[col + '_sin'] = np.sin(2 * np.pi * df[col] / max_val)
    df[col + '_cos'] = np.cos(2 * np.pi * df[col] / max_val)
    return df

train_df = encode_cyclical(train_df, 'Hour', 24)
train_df = encode_cyclical(train_df, 'DayOfWeek', 7)
train_df = encode_cyclical(train_df, 'Month', 12)
train_df = encode_cyclical(train_df, 'WeekOfYear', 52)

val_df = encode_cyclical(val_df, 'Hour', 24)
val_df = encode_cyclical(val_df, 'DayOfWeek', 7)
val_df = encode_cyclical(val_df, 'Month', 12)
val_df = encode_cyclical(val_df, 'WeekOfYear', 52)

test_df = encode_cyclical(test_df, 'Hour', 24)
test_df = encode_cyclical(test_df, 'DayOfWeek', 7)
test_df = encode_cyclical(test_df, 'Month', 12)
test_df = encode_cyclical(test_df, 'WeekOfYear', 52)

print("Applied cyclical encoding to time-based features.")

# --- 3.4 Exploratory Data Analysis (EDA) ---
# Visualize the relationship between features and the target variable.

# Plot 1: Average kWh by Hour of the Day
plt.figure(figsize=(12, 6))
sns.boxplot(data=train_df, x='Hour', y='Total_kWh', color='skyblue')
plt.title('Average Hourly Energy Consumption (Total_kWh)')
plt.xlabel('Hour of Day')
plt.ylabel('Total kWh')
plt.grid(True)
save_publication_fig('Fig02', 'hourly_energy_consumption_patterns')
plt.show()

# Plot 2: Average kWh by Day of the Week
plt.figure(figsize=(12, 6))
sns.boxplot(data=train_df, x='DayOfWeek', y='Total_kWh', color='lightgreen')
plt.title('Average Daily Energy Consumption (Total_kWh)')
plt.xlabel('Day of Week (0=Monday, 6=Sunday)')
plt.ylabel('Total kWh')
plt.grid(True)
save_publication_fig('Fig03', 'weekday_energy_consumption_patterns')
plt.show()

# SECTION 4: PREPARING DATA FOR TIME-SERIES MODELS

In [ ]:
# =============================================================================
# SECTION 4: PREPARING DATA FOR TIME-SERIES MODELS
# =============================================================================

# --- 4.1 Isolate Features and Target ---
feature_columns = [
    'Hour_sin', 'Hour_cos',
    'DayOfWeek_sin', 'DayOfWeek_cos',
    'Month_sin', 'Month_cos',
    'WeekOfYear_sin', 'WeekOfYear_cos',
    'IsWeekend', 'IsHoliday',
    'Outdoor_Temp', 'Outdoor_RH', 'Outdoor_WindSpeed',
    'Indoor_Temp_Avg', 'Indoor_RH_Avg', 'Indoor_Lux_Avg',
    'Indoor_Temp_mask', 'Indoor_RH_mask', 'Indoor_Lux_mask',
    'Total_kWh'
]
target_column = 'Total_kWh'

X_train_df = train_df[feature_columns]
y_train_df = train_df[[target_column]]
X_val_df = val_df[feature_columns]
y_val_df = val_df[[target_column]]
X_test_df = test_df[feature_columns]
y_test_df = test_df[[target_column]]

# --- 4.2 Feature Scaling (Corrected Workflow) ---
scaler_features = MinMaxScaler()
scaler_target = MinMaxScaler()

X_train_scaled = scaler_features.fit_transform(X_train_df)
y_train_scaled = scaler_target.fit_transform(y_train_df)

X_val_scaled = scaler_features.transform(X_val_df)
y_val_scaled = scaler_target.transform(y_val_df)
X_test_scaled = scaler_features.transform(X_test_df)
y_test_scaled = scaler_target.transform(y_test_df)

print("Data has been scaled correctly without leakage.")
print(f"Using {len(feature_columns)} input features for the probabilistic benchmark.")
print(feature_columns)

# --- 4.3 Create Sequences for One-Step-Ahead Forecasting ---
TIME_STEPS = 24

X_train_1step, y_train_1step = create_sequences(X_train_scaled, y_train_scaled.ravel(), TIME_STEPS)
X_val_1step, y_val_1step = create_sequences(X_val_scaled, y_val_scaled.ravel(), TIME_STEPS)
X_test_1step, y_test_1step = create_sequences(X_test_scaled, y_test_scaled.ravel(), TIME_STEPS)

print(f"\nCreated sequences for one-step-ahead forecasting with a look-back window of {TIME_STEPS} hours.")
print(f'X_train_1step shape: {X_train_1step.shape}')
print(f'y_train_1step shape: {y_train_1step.shape}')
print(f'X_val_1step shape: {X_val_1step.shape}')
print(f'y_val_1step shape: {y_val_1step.shape}')
print(f'X_test_1step shape: {X_test_1step.shape}')
print(f'y_test_1step shape: {y_test_1step.shape}')


# SECTION 5: MODEL 1 - BASELINE LSTM (ONE-STEP-AHEAD)

In [ ]:
# # =============================================================================
# # SECTION 5: MODEL 1 - HYPERPARAMETER TUNING FOR BASELINE LSTM
# # =============================================================================

# # --- 5.1 Import Modeling and Utility Libraries ---
# import setuptools
# import tensorflow as tf
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
# import keras_tuner as kt
# import time

# tf.random.set_seed(42)
# np.random.seed(42)

# # --- 5.2 Define the LSTM HyperModel for KerasTuner ---
# class LSTMHyperModel(kt.HyperModel):
#     def __init__(self, input_shape):
#         self.input_shape = input_shape

#     def build(self, hp):
#         """Builds a tunable probabilistic LSTM model."""
#         model = Sequential(name='Baseline_LSTM')

#         hp_units_1 = hp.Int('units_1', min_value=50, max_value=200, step=50)
#         hp_units_2 = hp.Int('units_2', min_value=50, max_value=200, step=50)
#         hp_dropout = hp.Float('dropout', min_value=0.2, max_value=0.5, step=0.1)
#         hp_learning_rate = hp.Choice('learning_rate', values=[1e-3, 5e-4, 1e-4])

#         model.add(LSTM(units=hp_units_1, return_sequences=True, input_shape=self.input_shape))
#         model.add(Dropout(hp_dropout))
#         model.add(LSTM(units=hp_units_2, return_sequences=False))
#         model.add(Dropout(hp_dropout))
#         model.add(Dense(3, name='quantile_output'))

#         model.compile(optimizer=Adam(learning_rate=hp_learning_rate), loss=quantile_loss)
#         return model

# # --- 5.3 Run the Bayesian Optimization Search ---
# input_shape = (X_train_1step.shape[1], X_train_1step.shape[2])
# hypermodel_lstm = LSTMHyperModel(input_shape)

# tuner_lstm = kt.BayesianOptimization(
#     hypermodel_lstm,
#     objective='val_loss',
#     max_trials=10,
#     executions_per_trial=1,
#     directory=str(HYPERPARAMETER_TUNING_DIR),
#     project_name='lstm_tuning_quantile',
#     overwrite=True,
#     seed=42
# )

# print("\n--- Starting Hyperparameter Search for Baseline LSTM Model ---")
# tuner_lstm.search(
#     X_train_1step, y_train_1step,
#     epochs=25,
#     batch_size=64,
#     validation_data=(X_val_1step, y_val_1step),
#     callbacks=[EarlyStopping(monitor='val_loss', patience=5)],
#     verbose=1
# )

# # --- 5.4 Train the Best LSTM Model and Evaluate ---
# best_hps_lstm = tuner_lstm.get_best_hyperparameters(num_trials=1)[0]
# print(f"""
# --- Optimal Hyperparameters for LSTM Found ---
# Layer 1 Units: {best_hps_lstm.get('units_1')}
# Layer 2 Units: {best_hps_lstm.get('units_2')}
# Dropout Rate: {best_hps_lstm.get('dropout'):.2f}
# Learning Rate: {best_hps_lstm.get('learning_rate')}
# """)

# best_lstm_model = tuner_lstm.hypermodel.build(best_hps_lstm)

# print("\n--- Training the Best Baseline LSTM Model ---")
# start_time = time.time()
# history_lstm = best_lstm_model.fit(
#     X_train_1step, y_train_1step,
#     epochs=100,
#     batch_size=64,
#     validation_data=(X_val_1step, y_val_1step),
#     callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
#                ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-5)],
#     verbose=1
# )

# results_lstm = evaluate_model(
#     'Tuned Baseline LSTM',
#     history_lstm,
#     best_lstm_model,
#     X_test_1step,
#     y_test_1step,
#     scaler_target,
#     start_time
# )


In [ ]:
# =============================================================================
# SECTION 5: MODEL 1 - FIXED BEST BASELINE LSTM (SKIP TUNING)
# =============================================================================

# --- 5.1 Import Modeling and Utility Libraries ---
import json
import time
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import setuptools
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.optimizers import Adam

tf.random.set_seed(42)
np.random.seed(42)

# --- 5.2 Robust Project Path Setup ---
# This makes the cell work whether you launch the notebook from the repo root
# or from the notebooks/ folder.
try:
    PROJECT_ROOT
except NameError:
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

OUTPUTS_MODELS_DIR = PROJECT_ROOT / "outputs" / "models"
OUTPUTS_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# --- 5.3 Locked Best Hyperparameters from Previous Tuning ---
BEST_LSTM_PARAMS = {
    "units_1": 200,
    "units_2": 100,
    "dropout": 0.20,
    "learning_rate": 0.001,
}

print(f"""
--- Optimal Hyperparameters for LSTM Found ---
Layer 1 Units: {BEST_LSTM_PARAMS['units_1']}
Layer 2 Units: {BEST_LSTM_PARAMS['units_2']}
Dropout Rate: {BEST_LSTM_PARAMS['dropout']:.2f}
Learning Rate: {BEST_LSTM_PARAMS['learning_rate']}
""")

# --- 5.4 Build the Locked Best LSTM Architecture ---
def build_fixed_best_lstm(input_shape):
    model = Sequential(name="Baseline_LSTM")
    model.add(Input(shape=input_shape))
    model.add(LSTM(units=BEST_LSTM_PARAMS["units_1"], return_sequences=True))
    model.add(Dropout(BEST_LSTM_PARAMS["dropout"]))
    model.add(LSTM(units=BEST_LSTM_PARAMS["units_2"], return_sequences=False))
    model.add(Dropout(BEST_LSTM_PARAMS["dropout"]))
    model.add(Dense(3, name="quantile_output"))
    model.compile(
        optimizer=Adam(learning_rate=BEST_LSTM_PARAMS["learning_rate"]),
        loss=quantile_loss,
    )
    return model

# --- 5.5 Model and History Paths ---
input_shape = (X_train_1step.shape[1], X_train_1step.shape[2])

model_path = OUTPUTS_MODELS_DIR / "best_lstm_fixed.keras"
history_path = OUTPUTS_MODELS_DIR / "best_lstm_fixed_history.csv"
metadata_path = OUTPUTS_MODELS_DIR / "best_lstm_fixed_metadata.json"

# --- 5.6 Load Saved Final Model If Present, Otherwise Train It ---
if model_path.exists() and history_path.exists() and metadata_path.exists():
    print(f"\n--- Loading Saved Best Baseline LSTM Model from {model_path} ---")

    best_lstm_model = load_model(
        model_path,
        custom_objects={"quantile_loss": quantile_loss},
    )

    history_df = pd.read_csv(history_path)
    history_lstm = SimpleNamespace(
        history={
            "loss": history_df["loss"].tolist(),
            "val_loss": history_df["val_loss"].tolist(),
        }
    )

    with metadata_path.open("r", encoding="utf-8") as handle:
        lstm_metadata = json.load(handle)

    # Make evaluate_model print the original stored training-time scale.
    start_time = time.time() - float(lstm_metadata.get("training_time_s", 0.0))

else:
    best_lstm_model = build_fixed_best_lstm(input_shape)

    print("\n--- Training the Best Baseline LSTM Model ---")
    start_time = time.time()

    history_lstm = best_lstm_model.fit(
        X_train_1step,
        y_train_1step,
        epochs=100,
        batch_size=64,
        validation_data=(X_val_1step, y_val_1step),
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
            ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=5, min_lr=1e-5),
        ],
        verbose=1,
    )

    training_time_s = time.time() - start_time

    best_lstm_model.save(model_path)

    pd.DataFrame(
        {
            "loss": history_lstm.history["loss"],
            "val_loss": history_lstm.history["val_loss"],
        }
    ).to_csv(history_path, index=False)

    with metadata_path.open("w", encoding="utf-8") as handle:
        json.dump(
            {
                "model": "Tuned Baseline LSTM",
                "training_time_s": training_time_s,
                "params": BEST_LSTM_PARAMS,
            },
            handle,
            indent=2,
        )

    print(f"\nSaved fixed best LSTM model to: {model_path}")
    print(f"Saved fixed best LSTM history to: {history_path}")

# --- 5.7 Evaluate ---
results_lstm = evaluate_model(
    "Tuned Baseline LSTM",
    history_lstm,
    best_lstm_model,
    X_test_1step,
    y_test_1step,
    scaler_target,
    start_time,
)


# SECTION 6: MODEL 2 - HYBRID CNN-LSTM

In [ ]:
# # =============================================================================
# # SECTION 6: MODEL 2 - HYPERPARAMETER TUNING FOR CNN-LSTM
# # =============================================================================

# # --- 6.1 Import Additional Libraries ---
# from tensorflow.keras.layers import Conv1D, Reshape
# import keras_tuner as kt

# # --- 6.2 Define the CNN-LSTM HyperModel for KerasTuner (Definitive Version) ---
# class CNNLSTMHyperModel(kt.HyperModel):
#     def __init__(self, input_shape):
#         self.input_shape = input_shape

#     def build(self, hp):
#         """Builds a tunable and robust probabilistic CNN-LSTM model."""
#         model = Sequential(name='CNN_LSTM')

#         hp_filters = hp.Int('filters', min_value=64, max_value=128, step=32)
#         hp_lstm_units_1 = hp.Int('lstm_units_1', min_value=100, max_value=200, step=50)
#         hp_lstm_units_2 = hp.Int('lstm_units_2', min_value=50, max_value=150, step=50)
#         hp_dropout = hp.Float('dropout', min_value=0.2, max_value=0.4, step=0.1)
#         hp_learning_rate = hp.Choice('learning_rate', values=[1e-3, 5e-4])

#         model.add(Conv1D(filters=hp_filters, kernel_size=3, activation='relu', input_shape=self.input_shape, padding='same'))
#         model.add(LSTM(units=hp_lstm_units_1, return_sequences=True))
#         model.add(Dropout(hp_dropout))
#         model.add(LSTM(units=hp_lstm_units_2, return_sequences=False))
#         model.add(Dropout(hp_dropout))
#         model.add(Dense(3, name='quantile_output'))

#         model.compile(optimizer=Adam(learning_rate=hp_learning_rate), loss=quantile_loss)
#         return model

# # --- 6.3 Run the Bayesian Optimization Search ---
# hypermodel_cnn_lstm = CNNLSTMHyperModel(input_shape)

# tuner_cnn_lstm = kt.BayesianOptimization(
#     hypermodel_cnn_lstm,
#     objective='val_loss',
#     max_trials=10,
#     executions_per_trial=1,
#     directory=str(HYPERPARAMETER_TUNING_DIR),
#     project_name='cnn_lstm_tuning_quantile',
#     overwrite=True,
#     seed=42
# )

# print("\n--- Starting Hyperparameter Search for CNN-LSTM Model ---")
# tuner_cnn_lstm.search(
#     X_train_1step, y_train_1step,
#     epochs=25,
#     batch_size=64,
#     validation_data=(X_val_1step, y_val_1step),
#     callbacks=[EarlyStopping(monitor='val_loss', patience=5)],
#     verbose=1
# )

# # --- 6.4 Train the Best CNN-LSTM Model and Evaluate ---
# best_hps_cnn_lstm = tuner_cnn_lstm.get_best_hyperparameters(num_trials=1)[0]
# print(f"""
# --- Optimal Hyperparameters for CNN-LSTM Found ---
# CNN Filters: {best_hps_cnn_lstm.get('filters')}
# LSTM Units 1: {best_hps_cnn_lstm.get('lstm_units_1')}
# LSTM Units 2: {best_hps_cnn_lstm.get('lstm_units_2')}
# Dropout Rate: {best_hps_cnn_lstm.get('dropout'):.2f}
# Learning Rate: {best_hps_cnn_lstm.get('learning_rate')}
# """)

# CHAMPION_CNN_LSTM_PARAMS = {
#     'filters': best_hps_cnn_lstm.get('filters'),
#     'lstm_units_1': best_hps_cnn_lstm.get('lstm_units_1'),
#     'lstm_units_2': best_hps_cnn_lstm.get('lstm_units_2'),
#     'dropout': best_hps_cnn_lstm.get('dropout'),
#     'learning_rate': best_hps_cnn_lstm.get('learning_rate'),
# }

# def build_probabilistic_champion_cnn_lstm(input_shape, output_horizons=1, model_name='Champion_CNN_LSTM'):
#     """Build the CNN-LSTM champion with probabilistic outputs."""
#     model = Sequential(name=model_name)
#     model.add(Conv1D(
#         filters=CHAMPION_CNN_LSTM_PARAMS['filters'],
#         kernel_size=3,
#         activation='relu',
#         input_shape=input_shape,
#         padding='same',
#     ))
#     model.add(LSTM(CHAMPION_CNN_LSTM_PARAMS['lstm_units_1'], return_sequences=True))
#     model.add(Dropout(CHAMPION_CNN_LSTM_PARAMS['dropout']))
#     model.add(LSTM(CHAMPION_CNN_LSTM_PARAMS['lstm_units_2'], return_sequences=False))
#     model.add(Dropout(CHAMPION_CNN_LSTM_PARAMS['dropout']))
#     model.add(Dense(output_horizons * 3, name='quantile_output'))

#     if output_horizons > 1:
#         model.add(Reshape((output_horizons, 3), name='quantile_horizon_output'))

#     model.compile(
#         optimizer=Adam(learning_rate=CHAMPION_CNN_LSTM_PARAMS['learning_rate']),
#         loss=quantile_loss,
#     )
#     return model

# best_cnn_lstm_model = build_probabilistic_champion_cnn_lstm(
#     input_shape,
#     output_horizons=1,
#     model_name='CNN_LSTM_Champion_One_Step',
# )

# print("\n--- Training the Best CNN-LSTM Model ---")
# start_time = time.time()
# history_cnn_lstm = best_cnn_lstm_model.fit(
#     X_train_1step, y_train_1step,
#     epochs=100,
#     batch_size=64,
#     validation_data=(X_val_1step, y_val_1step),
#     callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
#                ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-5)],
#     verbose=1
# )

# results_cnn_lstm = evaluate_model(
#     'Tuned CNN-LSTM',
#     history_cnn_lstm,
#     best_cnn_lstm_model,
#     X_test_1step,
#     y_test_1step,
#     scaler_target,
#     start_time
# )

In [ ]:
# =============================================================================
# SECTION 6: MODEL 2 - FIXED BEST CNN-LSTM (SKIP TUNING)
# =============================================================================

# --- 6.1 Import Additional Libraries ---
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout, Reshape, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import time
from pathlib import Path

tf.random.set_seed(42)
np.random.seed(42)

# --- 6.2 Locked Best Hyperparameters from Previous Tuning ---
CHAMPION_CNN_LSTM_PARAMS = {
    'filters': 96,
    'lstm_units_1': 200,
    'lstm_units_2': 100,
    'dropout': 0.40,
    'learning_rate': 0.001,
}

print(f"""
--- Optimal Hyperparameters for CNN-LSTM Found ---
CNN Filters: {CHAMPION_CNN_LSTM_PARAMS['filters']}
LSTM Units 1: {CHAMPION_CNN_LSTM_PARAMS['lstm_units_1']}
LSTM Units 2: {CHAMPION_CNN_LSTM_PARAMS['lstm_units_2']}
Dropout Rate: {CHAMPION_CNN_LSTM_PARAMS['dropout']:.2f}
Learning Rate: {CHAMPION_CNN_LSTM_PARAMS['learning_rate']}
""")

# --- 6.3 Build the Locked Best CNN-LSTM Architecture ---
def build_probabilistic_champion_cnn_lstm(input_shape, output_horizons=1, model_name='Champion_CNN_LSTM'):
    """Build the fixed best CNN-LSTM champion with probabilistic outputs."""
    model = Sequential(name=model_name)
    model.add(Input(shape=input_shape))
    model.add(Conv1D(
        filters=CHAMPION_CNN_LSTM_PARAMS['filters'],
        kernel_size=3,
        activation='relu',
        padding='same',
    ))
    model.add(LSTM(CHAMPION_CNN_LSTM_PARAMS['lstm_units_1'], return_sequences=True))
    model.add(Dropout(CHAMPION_CNN_LSTM_PARAMS['dropout']))
    model.add(LSTM(CHAMPION_CNN_LSTM_PARAMS['lstm_units_2'], return_sequences=False))
    model.add(Dropout(CHAMPION_CNN_LSTM_PARAMS['dropout']))
    model.add(Dense(output_horizons * 3, name='quantile_output'))

    if output_horizons > 1:
        model.add(Reshape((output_horizons, 3), name='quantile_horizon_output'))

    model.compile(
        optimizer=Adam(learning_rate=CHAMPION_CNN_LSTM_PARAMS['learning_rate']),
        loss=quantile_loss,
    )
    return model

# --- 6.4 Load Saved Final Model If Present, Otherwise Train It ---
input_shape = (X_train_1step.shape[1], X_train_1step.shape[2])
model_path = OUTPUTS_MODELS_DIR / 'best_cnn_lstm_fixed.keras'

if model_path.exists():
    print(f"\n--- Loading Saved Best CNN-LSTM Model from {model_path} ---")
    best_cnn_lstm_model = load_model(model_path, custom_objects={'quantile_loss': quantile_loss})
    history_cnn_lstm = None
    start_time = time.time()
else:
    best_cnn_lstm_model = build_probabilistic_champion_cnn_lstm(
        input_shape,
        output_horizons=1,
        model_name='CNN_LSTM_Champion_One_Step',
    )

    print("\n--- Training the Best CNN-LSTM Model ---")
    start_time = time.time()
    history_cnn_lstm = best_cnn_lstm_model.fit(
        X_train_1step, y_train_1step,
        epochs=100,
        batch_size=64,
        validation_data=(X_val_1step, y_val_1step),
        callbacks=[
            EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-5),
        ],
        verbose=1
    )

    best_cnn_lstm_model.save(model_path)
    print(f"\nSaved fixed best CNN-LSTM model to: {model_path}")

# --- 6.5 Evaluate ---
results_cnn_lstm = evaluate_model(
    'Tuned CNN-LSTM',
    history_cnn_lstm,
    best_cnn_lstm_model,
    X_test_1step,
    y_test_1step,
    scaler_target,
    start_time
)


# SECTION 7: MODEL 3 - TRANSFORMER AND FINAL COMPARISON (ONE-STEP-AHEAD)

In [ ]:
# # =============================================================================
# # SECTION 7: MODEL 3 - HYPERPARAMETER TUNING FOR TRANSFORMER (REVISED)
# # =============================================================================

# # --- 7.1 Import Additional Libraries ---
# from tensorflow.keras.layers import Input, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D, Add
# from tensorflow.keras.models import Model
# import keras_tuner as kt
# import time

# # --- 7.2 Define the Transformer HyperModel with Probabilistic Output ---
# class TransformerHyperModel(kt.HyperModel):
#     def __init__(self, input_shape):
#         self.input_shape = input_shape

#     def build(self, hp):
#         """Builds a tunable probabilistic Transformer model."""
#         d_model = hp.Int('d_model', min_value=32, max_value=128, step=32)
#         num_heads = hp.Int('num_heads', min_value=2, max_value=8, step=2)
#         ff_dim = hp.Int('ff_dim', min_value=64, max_value=256, step=64)
#         num_transformer_blocks = hp.Int('num_blocks', min_value=1, max_value=3, step=1)
#         dropout_rate = hp.Float('dropout', min_value=0.1, max_value=0.4, step=0.1)
#         learning_rate = hp.Choice('learning_rate', values=[1e-3, 5e-4, 1e-4])

#         inputs = Input(shape=self.input_shape)
#         x = Dense(d_model)(inputs)
#         x = PositionalEncoding(d_model)(x)

#         for _ in range(num_transformer_blocks):
#             attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(x, x)
#             attention_output = Dropout(dropout_rate)(attention_output)
#             x = Add()([x, attention_output])
#             x = LayerNormalization(epsilon=1e-6)(x)

#             ffn_output = Dense(ff_dim, activation='relu')(x)
#             ffn_output = Dense(d_model)(ffn_output)
#             ffn_output = Dropout(dropout_rate)(ffn_output)
#             x = Add()([x, ffn_output])
#             x = LayerNormalization(epsilon=1e-6)(x)

#         x = GlobalAveragePooling1D()(x)
#         x = Dropout(dropout_rate)(x)
#         outputs = Dense(3, name='quantile_output')(x)

#         model = Model(inputs=inputs, outputs=outputs, name='Transformer')
#         model.compile(optimizer=Adam(learning_rate=learning_rate), loss=quantile_loss)
#         return model

# # --- 7.3 Run the Bayesian Optimization Search ---
# hypermodel_transformer = TransformerHyperModel(input_shape)

# tuner_transformer = kt.BayesianOptimization(
#     hypermodel_transformer,
#     objective='val_loss',
#     max_trials=10,
#     executions_per_trial=1,
#     directory=str(HYPERPARAMETER_TUNING_DIR),
#     project_name='transformer_tuning_quantile',
#     overwrite=True,
#     seed=42
# )

# print("\n--- Starting Hyperparameter Search for Transformer Model ---")
# tuner_transformer.search(
#     X_train_1step, y_train_1step,
#     epochs=25,
#     batch_size=64,
#     validation_data=(X_val_1step, y_val_1step),
#     callbacks=[EarlyStopping(monitor='val_loss', patience=5)],
#     verbose=1
# )

# # --- 7.4 Train the Best Model and Evaluate ---
# best_hps_transformer = tuner_transformer.get_best_hyperparameters(num_trials=1)[0]
# print(f"""
# --- Optimal Hyperparameters for Transformer Found ---
# Model Dimension (d_model): {best_hps_transformer.get('d_model')}
# Number of Attention Heads: {best_hps_transformer.get('num_heads')}
# Feed-Forward Dimension: {best_hps_transformer.get('ff_dim')}
# Number of Transformer Blocks: {best_hps_transformer.get('num_blocks')}
# Dropout Rate: {best_hps_transformer.get('dropout'):.2f}
# Learning Rate: {best_hps_transformer.get('learning_rate')}
# """)

# best_transformer_model = tuner_transformer.hypermodel.build(best_hps_transformer)

# print("\n--- Training the Best Transformer Model ---")
# start_time = time.time()
# history_transformer = best_transformer_model.fit(
#     X_train_1step, y_train_1step,
#     epochs=100,
#     batch_size=64,
#     validation_data=(X_val_1step, y_val_1step),
#     callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
#                ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-5)],
#     verbose=1
# )

# results_transformer = evaluate_model(
#     'Tuned Transformer',
#     history_transformer,
#     best_transformer_model,
#     X_test_1step,
#     y_test_1step,
#     scaler_target,
#     start_time
# )

# # --- 7.5 Final Model Comparison ---
# all_results = [results_lstm, results_cnn_lstm, results_transformer]
# results_df = pd.DataFrame(all_results).set_index('Model')
# save_metrics(results_df.reset_index(), 'probabilistic_one_step_metrics.csv')
# print("\n\n--- Final Probabilistic One-Step Model Comparison ---")
# print(results_df.sort_values(by='Test RMSE (kWh)'))

# cnn_lstm_champion_model = best_cnn_lstm_model
# one_step_champion_model = best_cnn_lstm_model
# one_step_champion_name = 'Tuned CNN-LSTM'
# print('\nCNN-LSTM has been locked as the downstream champion model for Sections 8-12.')

In [ ]:
# =============================================================================
# SECTION 7: MODEL 3 - FIXED BEST TRANSFORMER (SKIP TUNING)
# =============================================================================

# --- 7.1 Import Additional Libraries ---
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
    GlobalAveragePooling1D,
    Add,
    Reshape,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import time
from pathlib import Path

tf.random.set_seed(42)
np.random.seed(42)

# --- 7.2 Locked Best Hyperparameters from Previous Tuning ---
BEST_TRANSFORMER_PARAMS = {
    'd_model': 64,
    'num_heads': 4,
    'ff_dim': 256,
    'num_blocks': 2,
    'dropout': 0.20,
    'learning_rate': 5e-4,
}

print(f"""
--- Optimal Hyperparameters for Transformer Found ---
Model Dimension (d_model): {BEST_TRANSFORMER_PARAMS['d_model']}
Number of Attention Heads: {BEST_TRANSFORMER_PARAMS['num_heads']}
Feed-Forward Dimension: {BEST_TRANSFORMER_PARAMS['ff_dim']}
Number of Transformer Blocks: {BEST_TRANSFORMER_PARAMS['num_blocks']}
Dropout Rate: {BEST_TRANSFORMER_PARAMS['dropout']:.2f}
Learning Rate: {BEST_TRANSFORMER_PARAMS['learning_rate']}
""")

# --- 7.3 Build the Locked Best Transformer Architecture ---
def build_fixed_best_transformer(input_shape):
    inputs = Input(shape=input_shape)

    x = Dense(BEST_TRANSFORMER_PARAMS['d_model'])(inputs)
    x = PositionalEncoding(BEST_TRANSFORMER_PARAMS['d_model'])(x)

    for _ in range(BEST_TRANSFORMER_PARAMS['num_blocks']):
        attention_output = MultiHeadAttention(
            num_heads=BEST_TRANSFORMER_PARAMS['num_heads'],
            key_dim=BEST_TRANSFORMER_PARAMS['d_model']
        )(x, x)
        attention_output = Dropout(BEST_TRANSFORMER_PARAMS['dropout'])(attention_output)
        x = Add()([x, attention_output])
        x = LayerNormalization(epsilon=1e-6)(x)

        ffn_output = Dense(BEST_TRANSFORMER_PARAMS['ff_dim'], activation='relu')(x)
        ffn_output = Dense(BEST_TRANSFORMER_PARAMS['d_model'])(ffn_output)
        ffn_output = Dropout(BEST_TRANSFORMER_PARAMS['dropout'])(ffn_output)
        x = Add()([x, ffn_output])
        x = LayerNormalization(epsilon=1e-6)(x)

    x = GlobalAveragePooling1D()(x)
    x = Dropout(BEST_TRANSFORMER_PARAMS['dropout'])(x)
    outputs = Dense(3, name='quantile_output')(x)

    model = Model(inputs=inputs, outputs=outputs, name='Transformer')
    model.compile(
        optimizer=Adam(learning_rate=BEST_TRANSFORMER_PARAMS['learning_rate']),
        loss=quantile_loss
    )
    return model

# --- 7.4 Load Saved Final Model If Present, Otherwise Train It ---
input_shape = (X_train_1step.shape[1], X_train_1step.shape[2])
model_path = OUTPUTS_MODELS_DIR / 'best_transformer_fixed.keras'

if model_path.exists():
    print(f"\n--- Loading Saved Best Transformer Model from {model_path} ---")
    best_transformer_model = load_model(
        model_path,
        custom_objects={
            'quantile_loss': quantile_loss,
            'PositionalEncoding': PositionalEncoding,
        }
    )
    history_transformer = None
    start_time = time.time()
else:
    best_transformer_model = build_fixed_best_transformer(input_shape)

    print("\n--- Training the Best Transformer Model ---")
    start_time = time.time()
    history_transformer = best_transformer_model.fit(
        X_train_1step, y_train_1step,
        epochs=100,
        batch_size=64,
        validation_data=(X_val_1step, y_val_1step),
        callbacks=[
            EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-5),
        ],
        verbose=1
    )

    best_transformer_model.save(model_path)
    print(f"\nSaved fixed best Transformer model to: {model_path}")

# --- 7.5 Evaluate ---
results_transformer = evaluate_model(
    'Tuned Transformer',
    history_transformer,
    best_transformer_model,
    X_test_1step,
    y_test_1step,
    scaler_target,
    start_time
)

# --- 7.6 Final Model Comparison ---
all_results = [results_lstm, results_cnn_lstm, results_transformer]
results_df = pd.DataFrame(all_results).set_index('Model')
save_metrics(results_df.reset_index(), 'probabilistic_one_step_metrics.csv')

print("\n\n--- Final Probabilistic One-Step Model Comparison ---")
print(results_df.sort_values(by='Test RMSE (kWh)'))

cnn_lstm_champion_model = best_cnn_lstm_model
one_step_champion_model = best_cnn_lstm_model
one_step_champion_name = 'Tuned CNN-LSTM'
print('\nCNN-LSTM has been locked as the downstream champion model for Sections 8-12.')


# SECTION 8: PROBABILISTIC MULTI-HORIZON FORECASTING WITH THE CNN-LSTM CHAMPION

In [ ]:
# =============================================================================
# SECTION 8: PROBABILISTIC MULTI-HORIZON FORECASTING WITH THE CNN-LSTM CHAMPION
# =============================================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

HORIZONS = [1, 3, 6, 12, 24]

def create_multi_horizon_sequences(features, target, time_steps, horizons):
    """Create aligned multivariate sequences for multiple forecast horizons."""
    X, y = [], []
    max_horizon = max(horizons)

    for i in range(len(features) - time_steps - max_horizon + 1):
        X.append(features[i:(i + time_steps)])
        y.append([target[i + time_steps + horizon - 1] for horizon in horizons])

    return np.array(X), np.array(y)


# --- 8.1 Create Multi-Horizon Sequences ---
X_train_multi, y_train_multi = create_multi_horizon_sequences(
    X_train_scaled, y_train_scaled.ravel(), TIME_STEPS, HORIZONS
)
X_val_multi, y_val_multi = create_multi_horizon_sequences(
    X_val_scaled, y_val_scaled.ravel(), TIME_STEPS, HORIZONS
)
X_test_multi, y_test_multi = create_multi_horizon_sequences(
    X_test_scaled, y_test_scaled.ravel(), TIME_STEPS, HORIZONS
)

print('Created multi-horizon probabilistic sequences for the CNN-LSTM champion:')
print(f'X_train_multi shape: {X_train_multi.shape}')
print(f'y_train_multi shape: {y_train_multi.shape}')
print(f'X_val_multi shape: {X_val_multi.shape}')
print(f'y_val_multi shape: {y_val_multi.shape}')
print(f'X_test_multi shape: {X_test_multi.shape}')
print(f'y_test_multi shape: {y_test_multi.shape}')

# --- 8.2 Train the Probabilistic Multi-Horizon CNN-LSTM Champion ---
input_shape_multi = (X_train_multi.shape[1], X_train_multi.shape[2])
multi_horizon_champion_model = build_probabilistic_champion_cnn_lstm(
    input_shape_multi,
    output_horizons=len(HORIZONS),
    model_name='Multi_Horizon_Champion_CNN_LSTM'
)

print()
print('--- Training the Probabilistic Multi-Horizon CNN-LSTM Champion ---')
start_time_multi_horizon = time.time()
history_multi_horizon = multi_horizon_champion_model.fit(
    X_train_multi,
    y_train_multi,
    epochs=100,
    batch_size=64,
    validation_data=(X_val_multi, y_val_multi),
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-5),
    ],
    verbose=1,
)
print(f"Completed multi-horizon CNN-LSTM training in {time.time() - start_time_multi_horizon:.2f} seconds.")

# --- 8.3 Evaluate the Probabilistic Multi-Horizon Forecasts ---
y_pred_multi_scaled = multi_horizon_champion_model.predict(X_test_multi, verbose=0)
y_pred_multi = scaler_target.inverse_transform(y_pred_multi_scaled.reshape(-1, 1)).reshape(y_pred_multi_scaled.shape)
y_test_multi_actual = scaler_target.inverse_transform(y_test_multi.reshape(-1, 1)).reshape(y_test_multi.shape)

multi_horizon_metrics = []
for horizon_idx, horizon in enumerate(HORIZONS):
    actual = y_test_multi_actual[:, horizon_idx]
    q05_pred = y_pred_multi[:, horizon_idx, 0]
    q50_pred = y_pred_multi[:, horizon_idx, 1]
    q95_pred = y_pred_multi[:, horizon_idx, 2]

    lower_interval = np.minimum(q05_pred, q95_pred)
    upper_interval = np.maximum(q05_pred, q95_pred)

    rmse = np.sqrt(mean_squared_error(actual, q50_pred))
    mae = mean_absolute_error(actual, q50_pred)
    r2 = r2_score(actual, q50_pred)
    picp = np.mean((actual >= lower_interval) & (actual <= upper_interval)) * 100
    mpiw = np.mean(upper_interval - lower_interval)

    multi_horizon_metrics.append({
        'Horizon (hours)': horizon,
        'RMSE (kWh)': rmse,
        'MAE (kWh)': mae,
        'R-squared': r2,
        'PICP (%)': picp,
        'MPIW (kWh)': mpiw,
    })

probabilistic_multi_horizon_df = pd.DataFrame(multi_horizon_metrics).set_index('Horizon (hours)')
save_metrics(probabilistic_multi_horizon_df.reset_index(), 'probabilistic_multi_horizon_metrics.csv')

print()
print('--- Probabilistic Multi-Horizon Forecast Performance (CNN-LSTM Champion) ---')
print(probabilistic_multi_horizon_df)

# --- 8.4 Plot Error Growth and Uncertainty Growth by Horizon ---
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

axes[0].plot(HORIZONS, probabilistic_multi_horizon_df['RMSE (kWh)'], marker='o', linewidth=2, label='RMSE')
axes[0].plot(HORIZONS, probabilistic_multi_horizon_df['MAE (kWh)'], marker='s', linewidth=2, label='MAE')
axes[0].set_title('Probabilistic Multi-Horizon Error Degradation (CNN-LSTM Champion)')
axes[0].set_ylabel('Error (kWh)')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(HORIZONS, probabilistic_multi_horizon_df['MPIW (kWh)'], marker='^', linewidth=2, color='#CC79A7')
axes[1].set_title('Prediction Interval Width Growth Over Forecast Horizon (CNN-LSTM Champion)')
axes[1].set_xlabel('Forecast Horizon (hours)')
axes[1].set_ylabel('MPIW (kWh)')
axes[1].grid(True)

plt.tight_layout()
save_publication_fig('Fig10', 'multi_horizon_probabilistic_degradation')
plt.show()

# SECTION 9: UNCERTAINTY-AWARE ANOMALY DETECTION WITH THE CNN-LSTM CHAMPION

In [ ]:
# =============================================================================
# SECTION 9: UNCERTAINTY-AWARE ANOMALY DETECTION WITH THE CNN-LSTM CHAMPION
# =============================================================================

# --- 9.1 Reuse or Rebuild the One-Step Probabilistic CNN-LSTM Champion ---
if 'best_cnn_lstm_model' in globals():
    one_step_champion_model = best_cnn_lstm_model
    print('Reusing the trained one-step CNN-LSTM champion model from Section 6.')
elif 'cnn_lstm_champion_model' in globals():
    one_step_champion_model = cnn_lstm_champion_model
    print('Reusing the CNN-LSTM champion model already in memory.')
else:
    print('Rebuilding and training the one-step CNN-LSTM champion for uncertainty-aware anomaly detection.')
    one_step_champion_model = build_probabilistic_champion_cnn_lstm(
        (X_train_1step.shape[1], X_train_1step.shape[2]),
        output_horizons=1,
        model_name='One_Step_Champion_CNN_LSTM',
    )
    history_one_step_champion = one_step_champion_model.fit(
        X_train_1step,
        y_train_1step,
        epochs=100,
        batch_size=64,
        validation_data=(X_val_1step, y_val_1step),
        callbacks=[
            EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-5),
        ],
        verbose=1,
    )

# --- 9.2 Generate Predictive Interval Bounds on the Test Set ---
y_pred_test_scaled = one_step_champion_model.predict(X_test_1step, verbose=0)
y_pred_test = scaler_target.inverse_transform(y_pred_test_scaled.reshape(-1, 1)).reshape(y_pred_test_scaled.shape)
y_actual_test = scaler_target.inverse_transform(y_test_1step.reshape(-1, 1)).ravel()

timestamps_test_1step = test_df.index[TIME_STEPS:]
q05_pred = y_pred_test[:, 0]
q50_pred = y_pred_test[:, 1]
q95_pred = y_pred_test[:, 2]

lower_bound = np.minimum(q05_pred, q95_pred)
upper_bound = np.maximum(q05_pred, q95_pred)

anomaly_flags = (y_actual_test > upper_bound) | (y_actual_test < lower_bound)
anomaly_timestamps = timestamps_test_1step[anomaly_flags]
anomaly_values = y_actual_test[anomaly_flags]

deviation_from_bounds = np.where(
    y_actual_test > upper_bound,
    y_actual_test - upper_bound,
    np.where(y_actual_test < lower_bound, lower_bound - y_actual_test, 0.0)
)

# --- 9.3 Plot the Uncertainty-Aware Anomaly Detection View ---
plt.figure(figsize=(16, 7))
plt.plot(timestamps_test_1step, y_actual_test, label='Actual kWh', color='#0072B2', linewidth=1.5, alpha=0.85)
plt.plot(timestamps_test_1step, q50_pred, label='Median Prediction (q50)', color='#D55E00', linestyle='--', linewidth=1.5)
plt.fill_between(
    timestamps_test_1step,
    lower_bound,
    upper_bound,
    color='#56B4E9',
    alpha=0.25,
    label='5th-95th Predictive Interval',
)
plt.plot(timestamps_test_1step, lower_bound, color='red', linestyle='--', linewidth=1.1, label='q05 Bound')
plt.plot(timestamps_test_1step, upper_bound, color='red', linestyle='--', linewidth=1.1, label='q95 Bound')
plt.scatter(anomaly_timestamps, anomaly_values, color='red', s=28, label='Detected Anomalies', zorder=5)
plt.title('Uncertainty-Aware Anomaly Detection with CNN-LSTM Predictive Intervals')
plt.xlabel('Timestamp')
plt.ylabel('Total kWh')
plt.legend(loc='upper right', ncol=2)
plt.grid(True)
save_publication_fig('Fig11', 'uncertainty_aware_anomalies')
plt.show()

# --- 9.4 Characterize Anomalies by Hour and Day of Week ---
anomaly_hours = pd.Series(anomaly_timestamps.hour).value_counts().reindex(range(24), fill_value=0).sort_index()
anomaly_days = pd.Series(anomaly_timestamps.dayofweek).value_counts().reindex(range(7), fill_value=0).sort_index()

plt.figure(figsize=(12, 5))
sns.barplot(x=anomaly_hours.index, y=anomaly_hours.values, color='#D55E00')
plt.title('CNN-LSTM Anomalies by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Anomaly Count')
plt.grid(True, axis='y')
save_publication_fig('Fig12', 'uncertainty_aware_anomaly_frequency_by_hour')
plt.show()

plt.figure(figsize=(12, 5))
sns.barplot(x=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], y=anomaly_days.values, color='#0072B2')
plt.title('CNN-LSTM Anomalies by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Anomaly Count')
plt.grid(True, axis='y')
save_publication_fig('Fig13', 'uncertainty_aware_anomaly_frequency_by_day')
plt.show()

# --- 9.5 Save Anomaly Summary Metrics ---
probabilistic_anomaly_summary_df = pd.DataFrame([
    {
        'Total Anomalies': int(anomaly_flags.sum()),
        'Anomaly Rate (%)': float(anomaly_flags.mean() * 100),
        'Average Deviation (kWh)': float(deviation_from_bounds[anomaly_flags].mean()) if anomaly_flags.any() else 0.0,
        'Maximum Deviation (kWh)': float(deviation_from_bounds[anomaly_flags].max()) if anomaly_flags.any() else 0.0,
    }
])
save_metrics(probabilistic_anomaly_summary_df, 'probabilistic_anomalies_summary.csv')

print()
print('--- Uncertainty-Aware Anomaly Summary (CNN-LSTM Champion) ---')
print(probabilistic_anomaly_summary_df)
print(f"Detected {int(anomaly_flags.sum())} anomalies using the CNN-LSTM 5th-95th predictive interval rule.")

# SECTION 10: EXPLAINABLE AI (SHAP ANALYSIS OF THE CNN-LSTM CHAMPION)

In [ ]:
# =============================================================================
# SECTION 10: EXPLAINABLE AI (SHAP ANALYSIS OF THE CNN-LSTM CHAMPION)
# =============================================================================

import shap
import tensorflow as tf
from tensorflow.keras.layers import Lambda
from tensorflow.keras.models import Model

if 'best_cnn_lstm_model' in globals():
    one_step_champion_model = best_cnn_lstm_model
elif 'cnn_lstm_champion_model' in globals():
    one_step_champion_model = cnn_lstm_champion_model
elif 'one_step_champion_model' not in globals():
    raise RuntimeError('Section 10 requires the probabilistic CNN-LSTM champion model from Section 9.')

# --- 10.1 Create a Median-Only View of the CNN-LSTM Champion Model ---
median_input = tf.keras.Input(
    shape=(X_train_1step.shape[1], X_train_1step.shape[2]),
    name='median_model_input',
)
quantile_output = one_step_champion_model(median_input)
median_output = Lambda(lambda x: x[:, 1], name='median_forecast')(quantile_output)
median_model = Model(
    inputs=median_input,
    outputs=median_output,
    name='CNN_LSTM_Champion_Median_Model',
)
_ = median_model(
    background_dataset[:1] if 'background_dataset' in globals() else tf.zeros((1, X_train_1step.shape[1], X_train_1step.shape[2]), dtype=tf.float32),
    training=False,
)

# --- 10.2 Sample Representative Background, Test, and Anomaly Sequences ---
rng = np.random.default_rng(42)
background_size = min(100, X_train_1step.shape[0])
test_sample_size = min(200, X_test_1step.shape[0])

background_indices = rng.choice(X_train_1step.shape[0], size=background_size, replace=False)
test_indices = rng.choice(X_test_1step.shape[0], size=test_sample_size, replace=False)

background_dataset = X_train_1step[background_indices].astype(np.float32)
test_shap_dataset = X_test_1step[test_indices].astype(np.float32)

anomaly_indices = np.flatnonzero(anomaly_flags)
if anomaly_indices.size == 0:
    raise RuntimeError('No anomalies were detected in Section 9, so CNN-LSTM SHAP anomaly explanation cannot proceed.')

anomaly_sequence_index = int(anomaly_indices[0])
anomaly_sequence = X_test_1step[anomaly_sequence_index:anomaly_sequence_index + 1].astype(np.float32)
anomaly_timestamp = timestamps_test_1step[anomaly_sequence_index]

print(f'Background SHAP dataset shape: {background_dataset.shape}')
print(f'Test SHAP dataset shape: {test_shap_dataset.shape}')
print(f'Explaining CNN-LSTM anomaly at timestamp: {anomaly_timestamp}')

# --- 10.3 Calculate SHAP Values with GradientExplainer ---
explainer = shap.GradientExplainer(median_model, background_dataset)

def unwrap_shap_values(raw_values):
    """Normalize SHAP outputs to (samples, timesteps, features)."""
    values = raw_values[0] if isinstance(raw_values, list) else raw_values
    values = np.asarray(values)

    while values.ndim > 3:
        if values.shape[-1] == 1:
            values = values[..., 0]
        elif values.shape[0] == 1:
            values = values[0]
        else:
            raise ValueError(f'Unexpected SHAP value shape: {values.shape}')
    return values


test_shap_values = unwrap_shap_values(explainer.shap_values(test_shap_dataset))
anomaly_shap_values = unwrap_shap_values(explainer.shap_values(anomaly_sequence))

# Sum across time to obtain feature-level attributions for standard SHAP plots.
test_shap_values_2d = test_shap_values.sum(axis=1)
test_inputs_2d = test_shap_dataset.sum(axis=1)
anomaly_shap_values_2d = anomaly_shap_values.sum(axis=1)
anomaly_inputs_2d = anomaly_sequence.sum(axis=1)

shap_feature_importance = pd.Series(
    np.abs(test_shap_values_2d).mean(axis=0),
    index=feature_columns,
).sort_values(ascending=False)

print()
print('Top 10 CNN-LSTM SHAP features (mean absolute attribution):')
print(shap_feature_importance.head(10))

# --- 10.4 Global SHAP Summary Plot ---
plt.figure(figsize=(12, 8))
shap.summary_plot(
    test_shap_values_2d,
    test_inputs_2d,
    feature_names=feature_columns,
    max_display=15,
    show=False,
)
save_publication_fig('Fig14', 'shap_global_summary')
plt.show()

# --- 10.5 Local SHAP Explanation for One Detected Anomaly ---
background_expected_value = median_model.predict(background_dataset, verbose=0).reshape(-1).mean()
anomaly_explanation = shap.Explanation(
    values=anomaly_shap_values_2d[0],
    base_values=background_expected_value,
    data=anomaly_inputs_2d[0],
    feature_names=feature_columns,
)

plt.figure(figsize=(12, 8))
shap.plots.waterfall(anomaly_explanation, max_display=15, show=False)
save_publication_fig('Fig15', 'shap_anomaly_explanation')
plt.show()

shap_top_3_features = shap_feature_importance.head(3).index.tolist()
print()
print('Top 3 CNN-LSTM SHAP features:')
for rank, feature_name in enumerate(shap_top_3_features, start=1):
    print(f'{rank}. {feature_name}')

# SECTION 11: COMMENTED-OUT REPRODUCTION BLOCK FOR STRICT FEW-SHOT TRANSFER LEARNING WITH THE CNN-LSTM CHAMPION

This section is intentionally commented out to prevent accidental execution of a long-running transfer-learning search.

When uncommented, it reproduces the full staged CNN-LSTM transfer-learning workflow from scratch by calling the dedicated script and regenerating the final metric tables.

Primary strict protocol:
- repeated non-overlapping few-shot windows on Floor 4
- external Floor 4 validation block used for recipe selection
- locked Floor 4 test block used only after the recipe was selected
- target scaler fit per few-shot window only
- feature-family-aware preprocessing in the strict protocol

The final strict recipe was not chosen upfront. It was selected in four stages by validation performance:
1. preprocessing comparison among `family_minmax`, `family_standard`, and `family_robust`
2. fine-tuning comparison among `head_only`, `last_recurrent_block`, `last_two_blocks`, and `full_model_tiny_lr`
3. optimization comparison among compact, validation-driven settings including `default`, `small_batch_longer_patience`, and `low_lr_clipped`
4. loss alignment comparison among `quantile_loss`, `weighted_median_quantile`, and `two_stage_weighted_then_quantile`

The resulting strict winner was:
- preprocessing: `family_standard`
- fine-tuning: `last_two_blocks`
- optimization: `small_batch_longer_patience`
- loss: `weighted_median_quantile`

The expected strict final test summary from the saved run was:
- `24.45 vs 35.12 kWh` at 1 day
- `21.38 vs 41.12 kWh` at 3 days
- `18.17 vs 36.17 kWh` at 7 days
- `17.17 vs 26.71 kWh` at 14 days

The authoritative strict outputs regenerated by the script are:
- `strict_protocol_stage_winners.csv`
- `strict_protocol_candidate_recipes.csv`
- `strict_best_recipe_per_window_test.csv`
- `strict_best_recipe_test_summary.csv`

In [ ]:
# =============================================================================
# SECTION 11: REPRODUCE STRICT FEW-SHOT TRANSFER LEARNING SEARCH
# COMMENTED OUT BY DEFAULT TO AVOID ACCIDENTAL LONG RUNS
# =============================================================================
#
# This section reproduces the full staged CNN-LSTM transfer-learning search
# from scratch by calling:
#   scripts/run_few_shot_protocol_search.py
#
# The script performs:
#   1. Stage 1 strict preprocessing / scaler comparison
#   2. Stage 2 transfer fine-tuning strategy comparison
#   3. Stage 3 compact optimization comparison
#   4. Stage 4 loss alignment comparison
#   5. Final strict test evaluation
#
# The currently locked strict winner from the saved run is:
#   family_standard + last_two_blocks + small_batch_longer_patience +
#   weighted_median_quantile
#
# Running the script regenerates these authoritative strict outputs:
#   - outputs/metrics/strict_protocol_stage_winners.csv
#   - outputs/metrics/strict_protocol_candidate_recipes.csv
#   - outputs/metrics/strict_best_recipe_per_window_test.csv
#   - outputs/metrics/strict_best_recipe_test_summary.csv
#
# The expected strict final test results from the saved run were:
#   - 1 day:  transfer 24.45 kWh vs scratch 35.12 kWh
#   - 3 days: transfer 21.38 kWh vs scratch 41.12 kWh
#   - 7 days: transfer 18.17 kWh vs scratch 36.17 kWh
#   - 14 days: transfer 17.17 kWh vs scratch 26.71 kWh
#
# To reproduce from scratch, deliberately uncomment and run this cell.
#
# from pathlib import Path
# import subprocess
# import pandas as pd
# import matplotlib.pyplot as plt
#
# project_root = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
# script_path = project_root / 'scripts' / 'run_few_shot_protocol_search.py'
#
# subprocess.run(
#     ['.venv/bin/python', str(script_path)],
#     cwd=project_root,
#     check=True,
# )
#
# metrics_dir = project_root / 'outputs' / 'metrics'
# strict_stage_winners_df = pd.read_csv(metrics_dir / 'strict_protocol_stage_winners.csv')
# strict_candidate_recipes_df = pd.read_csv(metrics_dir / 'strict_protocol_candidate_recipes.csv')
# strict_per_window_df = pd.read_csv(metrics_dir / 'strict_best_recipe_per_window_test.csv')
# strict_summary_df = pd.read_csv(metrics_dir / 'strict_best_recipe_test_summary.csv')
#
# display(strict_stage_winners_df)
# display(strict_candidate_recipes_df)
# display(strict_per_window_df)
# display(strict_summary_df)
#
# strict_budget_df = strict_summary_df.loc[strict_summary_df['Training Days'] != 'ALL'].copy()
# strict_budget_df['Training Days'] = strict_budget_df['Training Days'].astype(int)
# strict_budget_df.sort_values('Training Days', inplace=True)
#
# plt.figure(figsize=(12, 6))
# plt.plot(
#     strict_budget_df['Training Days'],
#     strict_budget_df['Scratch RMSE Mean (kWh)'],
#     marker='o',
#     linestyle='--',
#     color='red',
#     linewidth=2,
#     label='Train from Scratch (CNN-LSTM)',
# )
# plt.fill_between(
#     strict_budget_df['Training Days'],
#     strict_budget_df['Scratch RMSE Mean (kWh)'] - strict_budget_df['Scratch RMSE 95% CI Half Width (kWh)'],
#     strict_budget_df['Scratch RMSE Mean (kWh)'] + strict_budget_df['Scratch RMSE 95% CI Half Width (kWh)'],
#     color='red',
#     alpha=0.10,
# )
# plt.plot(
#     strict_budget_df['Training Days'],
#     strict_budget_df['Transfer RMSE Mean (kWh)'],
#     marker='o',
#     linestyle='-',
#     color='green',
#     linewidth=2,
#     label='Transfer Learning (Strict CNN-LSTM Recipe)',
# )
# plt.fill_between(
#     strict_budget_df['Training Days'],
#     strict_budget_df['Transfer RMSE Mean (kWh)'] - strict_budget_df['Transfer RMSE 95% CI Half Width (kWh)'],
#     strict_budget_df['Transfer RMSE Mean (kWh)'] + strict_budget_df['Transfer RMSE 95% CI Half Width (kWh)'],
#     color='green',
#     alpha=0.10,
# )
# plt.title('Final Strict Few-Shot Transfer Learning Curve (CNN-LSTM Champion)')
# plt.xlabel('Training Days')
# plt.ylabel('Test RMSE (kWh)')
# plt.xticks([1, 3, 7, 14])
# plt.grid(True)
# plt.legend()
#
# figures_dir = project_root / 'outputs' / 'figures'
# figures_dir.mkdir(parents=True, exist_ok=True)
# plt.savefig(figures_dir / 'Fig16_strict_final_transfer_learning.png', dpi=600, bbox_inches='tight')
# plt.savefig(figures_dir / 'Fig16_strict_final_transfer_learning.pdf', bbox_inches='tight')
# plt.show()


In [ ]:
# =============================================================================
# STRICT FINAL RESULTS ONLY
# Loads the persisted strict transfer-learning results and recreates the figure
# without rerunning the staged search.
# =============================================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
metrics_dir = project_root / "outputs" / "metrics"
figures_dir = project_root / "outputs" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

STRICT_BEST_RECIPE = {
    "Preprocess": "family_standard",
    "Fine-Tune Strategy": "last_two_blocks",
    "Optimization": "small_batch_longer_patience",
    "Loss Strategy": "weighted_median_quantile",
}

strict_stage_winners_df = pd.read_csv(metrics_dir / "strict_protocol_stage_winners.csv")
strict_summary_df = pd.read_csv(metrics_dir / "strict_best_recipe_test_summary.csv")

strict_budget_df = strict_summary_df.loc[strict_summary_df["Training Days"] != "ALL"].copy()
strict_budget_df["Training Days"] = strict_budget_df["Training Days"].astype(int)
strict_budget_df.sort_values("Training Days", inplace=True)

strict_overall_row = strict_summary_df.loc[strict_summary_df["Training Days"] == "ALL"].iloc[0]

print("--- Final Strict CNN-LSTM Transfer-Learning Recipe ---")
print(STRICT_BEST_RECIPE)
print()

print("--- Strict Stage Winners ---")
print(strict_stage_winners_df.to_string(index=False))
print()

print("--- Final Strict Test Summary ---")
print(strict_summary_df.to_string(index=False))
print()

print("Expected strict per-budget headline results:")
for _, row in strict_budget_df.iterrows():
    days = int(row["Training Days"])
    transfer_rmse = row["Transfer RMSE Mean (kWh)"]
    scratch_rmse = row["Scratch RMSE Mean (kWh)"]
    print(f"{days} day(s): transfer {transfer_rmse:.2f} kWh vs scratch {scratch_rmse:.2f} kWh")

print()
print(f"Overall strict transfer RMSE: {strict_overall_row['Transfer RMSE Mean (kWh)']:.4f} kWh")
print(f"Overall strict scratch RMSE: {strict_overall_row['Scratch RMSE Mean (kWh)']:.4f} kWh")
print(f"Overall paired improvement: {strict_overall_row['Paired Improvement Mean (kWh)']:.4f} kWh")

plt.figure(figsize=(12, 6))

plt.plot(
    strict_budget_df["Training Days"],
    strict_budget_df["Scratch RMSE Mean (kWh)"],
    marker="o",
    linestyle="--",
    color="red",
    linewidth=2,
    label="Train from Scratch (CNN-LSTM)",
)
plt.fill_between(
    strict_budget_df["Training Days"],
    strict_budget_df["Scratch RMSE Mean (kWh)"] - strict_budget_df["Scratch RMSE 95% CI Half Width (kWh)"],
    strict_budget_df["Scratch RMSE Mean (kWh)"] + strict_budget_df["Scratch RMSE 95% CI Half Width (kWh)"],
    color="red",
    alpha=0.10,
)

plt.plot(
    strict_budget_df["Training Days"],
    strict_budget_df["Transfer RMSE Mean (kWh)"],
    marker="o",
    linestyle="-",
    color="green",
    linewidth=2,
    label="Transfer Learning (Strict CNN-LSTM Recipe)",
)
plt.fill_between(
    strict_budget_df["Training Days"],
    strict_budget_df["Transfer RMSE Mean (kWh)"] - strict_budget_df["Transfer RMSE 95% CI Half Width (kWh)"],
    strict_budget_df["Transfer RMSE Mean (kWh)"] + strict_budget_df["Transfer RMSE 95% CI Half Width (kWh)"],
    color="green",
    alpha=0.10,
)

plt.title("Final Strict Few-Shot Transfer Learning Curve (CNN-LSTM Champion)")
plt.xlabel("Training Days")
plt.ylabel("Test RMSE (kWh)")
plt.xticks([1, 3, 7, 14])
plt.grid(True)
plt.legend()

png_path = figures_dir / "Fig16_strict_final_transfer_learning.png"
pdf_path = figures_dir / "Fig16_strict_final_transfer_learning.pdf"

plt.savefig(png_path, dpi=600, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")
plt.show()

print()
print(f"Saved 600 DPI PNG to: {png_path}")
print(f"Saved PDF to: {pdf_path}")


# SECTION 12: CNN-LSTM COMPUTE PROFILING

In [ ]:
# =============================================================================
# SECTION 12: CNN-LSTM COMPUTE PROFILING
# =============================================================================

cnn_lstm_profile_model = best_cnn_lstm_model if 'best_cnn_lstm_model' in globals() else one_step_champion_model

total_params = int(cnn_lstm_profile_model.count_params())
trainable_params = int(np.sum([np.prod(weight.shape) for weight in cnn_lstm_profile_model.trainable_weights]))
non_trainable_params = int(np.sum([np.prod(weight.shape) for weight in cnn_lstm_profile_model.non_trainable_weights]))

inference_start = time.perf_counter()
_ = cnn_lstm_profile_model.predict(X_test_1step, verbose=0)
total_inference_time = time.perf_counter() - inference_start
inference_latency_ms = (total_inference_time / len(X_test_1step)) * 1000

compute_profiling_df = pd.DataFrame([
    {
        'Model': 'Tuned CNN-LSTM Champion',
        'Total Parameters': total_params,
        'Trainable Parameters': trainable_params,
        'Non-Trainable Parameters': non_trainable_params,
        'Inference Time on Test Set (s)': float(total_inference_time),
        'Inference Latency (ms/sample)': float(inference_latency_ms),
        'Test Samples': int(len(X_test_1step)),
    }
])
save_metrics(compute_profiling_df, 'compute_profiling.csv')

print()
print('--- CNN-LSTM Compute Profiling ---')
print(compute_profiling_df)